In [47]:
# !pip install pip --upgrade


# QUBO hamiltoniano

In [48]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator

In [49]:
# --- PUNTO 1: MAPPATURA QUBO (Ottimizzazione Codebook) ---
# Definiamo un Hamiltoniano di Ising che rappresenta la funzione di costo della VQ.
# La relazione usata è xi = (I - Zi)/2 [7].
n_qubits = 4
# Esempio di termini: identità, interazioni locali (ZZ) e campi (Z) [7, 8]
coeffs = [1.0, 0.5, 0.5, -1.0]
operators = ["IIII", "IIIZ", "IIZI", "IIZZ"]
hamiltonian = SparsePauliOp(operators, coeffs=coeffs)

In [50]:
# --- PUNTO 2: DESIGN DELL'ANSATZ (Hardware-Efficient) ---
# Utilizziamo l'ansatz EfficientSU2 suggerito per i dispositivi NISQ [6].
# Include layer di rotazione RY/RZ e layer di entanglement CNOT [9].
ansatz = EfficientSU2(n_qubits, entanglement='linear', reps=2)
params = np.random.random(ansatz.num_parameters) # Parametri iniziali theta [10]

/tmp/ipykernel_55/3389527591.py:4: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n_qubits, entanglement='linear', reps=2)


In [51]:

# --- PUNTO 3: TRAINING (VQE Core) ---
# Eseguiamo il calcolo dell'energia (valore di aspettativa) [11].
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(ansatz)
# isa_circuit.measure_all()
isa_observable = hamiltonian.apply_layout(isa_circuit.layout)



In [52]:
estimator = Estimator()

job = estimator.run([(isa_circuit, isa_observable, params)])
pub_result = job.result()[0]
print(f"Expectation values: {pub_result.data.evs}")
# print(f"Energia misurata (Costo VQ): {result_vqe.data.evs}")[12]


Expectation values: 0.8302910375593617


**RICERCA CON GROVER (Encoding/Search)**

In [53]:
grover_circuit = QuantumCircuit(4)
grover_circuit.h([14-16]) # Sovrapposizione iniziale [1]

In [54]:
# Oracolo per lo stato '1011' [17, 18]
def oracle(qc):
    qc.x(2) # Inverte i bit che nel target sono 0 per usare mcx su '1111' [19]
    qc.h(3)
    qc.mcx([1,2], 3) # Inversione di fase della soluzione [20]
    qc.h(3)
    qc.x(2) # Undo inversione [18]
# Diffusore (Amplificazione della probabilità) [3, 21]
def diffuser(qc):
    qc.h([14-16])
    qc.x([14-16])
    qc.h(3)
    qc.mcx([14, 15], 3)
    qc.h(3)
    qc.x([14-16])
    qc.h([14-16])

In [55]:
oracle(grover_circuit)
diffuser(grover_circuit)
grover_circuit.measure_all()

CircuitError: 'Index 14 out of range for size 4.'

# COPILOT VQE

In [56]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from scipy.optimize import minimize

# =========================
# PUNTO 1: Hamiltoniano QUBO/Ising
# =========================
n_qubits = 4
coeffs = [1.0, 0.5, 0.5, -1.0]
operators = ["IIII", "IIIZ", "IIZI", "IIZZ"]
hamiltonian = SparsePauliOp(operators, coeffs=coeffs)

# =========================
# PUNTO 2: Ansatz
# =========================
ansatz = EfficientSU2(n_qubits, entanglement="linear", reps=2)
initial_params = np.random.random(ansatz.num_parameters)

# Backend + transpilation
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_circuit.layout)

# =========================
# PUNTO 3: VQE con ottimizzazione classica
# =========================
estimator = Estimator()

def cost_function(theta: np.ndarray) -> float:
    """Ritorna <H> per i parametri theta."""
    job = estimator.run([(isa_circuit, isa_observable, theta)])
    result = job.result()[0]
    return float(result.data.evs)

print("=== VQE ===")
initial_energy = cost_function(initial_params)
print(f"Energia iniziale: {initial_energy:.6f}")

opt_result = minimize(
    cost_function,
    x0=initial_params,
    method="COBYLA",
    options={"maxiter": 80}
)

best_theta = opt_result.x
best_energy = opt_result.fun
print(f"Energia finale ottimizzata: {best_energy:.6f}")
print(f"Convergenza: {opt_result.success}, msg: {opt_result.message}")

# =========================
# PUNTO 4: Grover (4 qubit) target = '1011'
# q3 q2 q1 q0 = 1 0 1 1
# =========================
grover = QuantumCircuit(4, 4)

# Sovrapposizione iniziale su tutti i qubit
grover.h([0, 1, 2, 3])

def oracle_target_1011(qc: QuantumCircuit):
    # target: q3=1, q2=0, q1=1, q0=1
    # Per marcare uno stato con un solo 0 (q2), faccio X su q2
    qc.x(2)

    # Multi-controlled Z usando H-MCX-H sul target q3 con controlli q0,q1,q2
    qc.h(3)
    qc.mcx([0, 1, 2], 3)
    qc.h(3)

    # Undo
    qc.x(2)

def diffuser(qc: QuantumCircuit):
    qc.h([0, 1, 2, 3])
    qc.x([0, 1, 2, 3])

    qc.h(3)
    qc.mcx([0, 1, 2], 3)
    qc.h(3)

    qc.x([0, 1, 2, 3])
    qc.h([0, 1, 2, 3])

# Numero iterazioni Grover ~ floor(pi/4*sqrt(N/M)), N=16, M=1 => ~3
num_iterations = 3
for _ in range(num_iterations):
    oracle_target_1011(grover)
    diffuser(grover)

# Misura
grover.measure([0, 1, 2, 3], [0, 1, 2, 3])

print("\n=== GROVER ===")
print(grover.draw(fold=120))

# Esecuzione con Sampler
sampler = Sampler()
job = sampler.run([grover], shots=2048)

# In SamplerV2 i counts sono in data.meas.get_counts()
# counts = res.meas.get_counts()
res = job.result()[0]
bits = res.data.c              # BitArray
counts = bits.get_counts()     # dizionario tipo {'1011': 1500, ...}
print(counts)
print("Most probable:", max(counts, key=counts.get))

# Stato più frequente
# most_probable = max(counts, key=counts.get)
# print("Stato più probabile:", most_probable)

/tmp/ipykernel_55/711191727.py:22: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n_qubits, entanglement="linear", reps=2)


=== VQE ===
Energia iniziale: 1.225352
Energia finale ottimizzata: -0.883796
Convergenza: False, msg: Return from COBYLA because the objective function has been evaluated MAXFUN times.

=== GROVER ===
     ┌───┐          ┌───┐┌───┐               ┌───┐┌───┐               ┌───┐┌───┐               ┌───┐┌───┐          »
q_0: ┤ H ├───────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├────────────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├──────────»
     ├───┤       │  ├───┤├───┤            │  ├───┤├───┤            │  ├───┤├───┤            │  ├───┤├───┤          »
q_1: ┤ H ├───────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├────────────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├──────────»
     ├───┤┌───┐  │  ├───┤├───┤┌───┐       │  ├───┤├───┤┌───┐       │  ├───┤├───┤┌───┐       │  ├───┤├───┤┌───┐     »
q_2: ┤ H ├┤ X ├──■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├─────»
     ├───┤├───┤┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├─

## scenario reale

In [4]:
import numpy as np
from itertools import product
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

# =========================================================
# SCENARIO REALE: Channel Selection (4 candidati)
# x_i in {0,1} indica se seleziono il canale i
#
# Costo:
#   C(x) = sum_i q_i x_i + sum_{i<j} J_ij x_i x_j + A*(sum_i x_i - K)^2
#
# q_i  : "costo locale" (rumore, occupazione, error rate)
# J_ij : interferenza tra coppie di canali scelti insieme
# K    : numero desiderato di canali attivi
# A    : peso del vincolo
# =========================================================

n = 4
K = 2
A = 3.0  # peso vincolo cardinalità

# Esempio dati "realistici" (da telemetria)
# più basso = meglio
q = np.array([0.8, 0.2, 1.1, 0.5])

# Interferenze simmetriche (0 sulla diagonale)
J = np.array([
    [0.0, 1.2, 0.4, 0.9],
    [1.2, 0.0, 0.7, 0.3],
    [0.4, 0.7, 0.0, 1.0],
    [0.9, 0.3, 1.0, 0.0],
])

# ---------- Utility: costruzione QUBO ----------
# C(x)=const + sum_i a_i x_i + sum_{i<j} b_ij x_i x_j
# con vincolo A*(sum x - K)^2
# => a_i += A*(1 - 2K), b_ij += 2A, const += A*K^2
a = q.astype(float).copy()
b = np.zeros((n, n), dtype=float)
const = A * (K ** 2)

for i in range(n):
    a[i] += A * (1 - 2 * K)

for i in range(n):
    for j in range(i + 1, n):
        b[i, j] += J[i, j] + 2 * A

# ---------- Mapping QUBO -> Ising ----------
# x_i = (1 - Z_i)/2
# x_i x_j = 1/4 (1 - Z_i - Z_j + Z_i Z_j)

h = np.zeros(n, dtype=float)             # coefficienti Z_i
Jz = np.zeros((n, n), dtype=float)       # coefficienti Z_i Z_j
c0 = const

# termini lineari
for i in range(n):
    c0 += a[i] / 2
    h[i] += -a[i] / 2

# termini quadratici
for i in range(n):
    for j in range(i + 1, n):
        cij = b[i, j]
        c0 += cij / 4
        h[i] += -cij / 4
        h[j] += -cij / 4
        Jz[i, j] += cij / 4

# ---------- Costruzione SparsePauliOp ----------
paulis = []
coeffs = []

# termine costante
paulis.append("I" * n)
coeffs.append(c0)

# termini Z_i
for i in range(n):
    label = ["I"] * n
    # convenzione stringa Pauli: sinistra=qubit alto, destra=qubit basso
    label[n - 1 - i] = "Z"
    paulis.append("".join(label))
    coeffs.append(h[i])

# termini Z_i Z_j
for i in range(n):
    for j in range(i + 1, n):
        if abs(Jz[i, j]) > 1e-12:
            label = ["I"] * n
            label[n - 1 - i] = "Z"
            label[n - 1 - j] = "Z"
            paulis.append("".join(label))
            coeffs.append(Jz[i, j])

hamiltonian = SparsePauliOp(paulis, coeffs=np.array(coeffs, dtype=float))

# ---------- VQE (solo ottimizzazione parametri ansatz) ----------
ansatz = EfficientSU2(n, entanglement="linear", reps=2)
theta = np.random.random(ansatz.num_parameters)

backend = AerSimulator()
estimator = Estimator()

# transpile circuito
t_ansatz = transpile(ansatz, backend=backend, optimization_level=1)
obs = hamiltonian.apply_layout(t_ansatz.layout)

def energy(params):
    job = estimator.run([(t_ansatz, obs, params)])
    return float(job.result()[0].data.evs)

# ottimizzazione semplice random-restart manuale (robusta senza scipy)
best_e = float("inf")
best_theta = None
for _ in range(30):
    cand = np.random.uniform(0, 2*np.pi, size=ansatz.num_parameters)
    e = energy(cand)
    if e < best_e:
        best_e = e
        best_theta = cand

print("Energia minima stimata (VQE-like):", best_e)

# ---------- Estrazione soluzione operativa ----------
# Per n=4 in produzione useresti un post-process più avanzato;
# qui enumeriamo tutte le bitstring per scegliere il costo minimo reale.
def classical_cost(x_bits):
    x = np.array(x_bits, dtype=float)
    lin = np.dot(q, x)
    quad = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            quad += J[i, j] * x[i] * x[j]
    penalty = A * (np.sum(x) - K) ** 2
    return lin + quad + penalty

all_x = list(product([0, 1], repeat=n))
best_x = min(all_x, key=classical_cost)
best_cost = classical_cost(best_x)

print("Miglior configurazione canali (x0..x3):", best_x)
print("Costo reale minimo:", best_cost)

selected = [i for i, bit in enumerate(best_x) if bit == 1]
print("Canali da attivare:", selected)

/tmp/ipykernel_56/3846577559.py:104: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n, entanglement="linear", reps=2)
/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


Energia minima stimata (VQE-like): 3.1153781766961695
Miglior configurazione canali (x0..x3): (0, 1, 0, 1)
Costo reale minimo: 1.0
Canali da attivare: [1, 3]


In [5]:
# import numpy as np

# def build_qubo(q, J, A, K, symmetric=False):
#     """
#     q: array (n,)
#     J: matrice (n,n), usa solo i<j
#     ritorna Q, c tali che min x^T Q x + c
#     """
#     n = len(q)
#     Q = np.zeros((n, n), dtype=float)

#     # coefficienti derivati
#     a = q + A * (1 - 2 * K)   # lineari
#     c = A * (K ** 2)          # costante

#     # diagonale (lineari)
#     for i in range(n):
#         Q[i, i] = a[i]

#     # off-diagonali (quadratici)
#     for i in range(n):
#         for j in range(i + 1, n):
#             bij = J[i, j] + 2 * A
#             if symmetric:
#                 Q[i, j] = bij / 2
#                 Q[j, i] = bij / 2
#             else:
#                 Q[i, j] = bij
#                 Q[j, i] = 0.0

#     return Q, c

In [6]:
# from itertools import product
# import numpy as np

# def original_cost(x, q, J, A, K):
#     x = np.array(x, dtype=float)
#     lin = np.dot(q, x)
#     quad = sum(J[i, j] * x[i] * x[j] for i in range(len(x)) for j in range(i+1, len(x)))
#     pen = A * (np.sum(x) - K) ** 2
#     return lin + quad + pen

# def qubo_cost(x, Q, c):
#     x = np.array(x, dtype=float)
#     return x @ Q @ x + c

# # esempio
# q = np.array([0.8, 0.2, 1.1, 0.5])
# J = np.array([
#     [0.0, 1.2, 0.4, 0.9],
#     [1.2, 0.0, 0.7, 0.3],
#     [0.4, 0.7, 0.0, 1.0],
#     [0.9, 0.3, 1.0, 0.0],
# ])
# A, K = 3.0, 2

# Q, c = build_qubo(q, J, A, K, symmetric=False)

# for x in product([0,1], repeat=4):
#     o = original_cost(x, q, J, A, K)
#     v = qubo_cost(x, Q, c)
#     assert abs(o - v) < 1e-9, (x, o, v)

# print("Verifica OK: forma QUBO equivalente alla funzione originale.")

Verifica OK: forma QUBO equivalente alla funzione originale.


In [9]:
# import numpy as np
# from qiskit.quantum_info import SparsePauliOp

# def qubo_to_sparse_pauli(Q: np.ndarray, c: float = 0.0, tol: float = 1e-12) -> SparsePauliOp:
#     """
#     Converte E(x)=x^T Q x + c in Hamiltoniano Ising H(z)=const + sum h_i Z_i + sum J_ij Z_i Z_j
#     con mapping x_i=(1-Z_i)/2.

#     Funziona con Q generale (non necessariamente simmetrica):
#     usa Qs=(Q+Q^T)/2 perché x^TQx dipende solo dalla parte simmetrica.
#     """
#     Q = np.array(Q, dtype=float)
#     n = Q.shape[0]
#     assert Q.shape == (n, n), "Q deve essere quadrata"

#     Qs = 0.5 * (Q + Q.T)  # parte simmetrica equivalente
#     const = float(c)
#     h = np.zeros(n, dtype=float)
#     Jzz = np.zeros((n, n), dtype=float)

#     # Diagonale: Qii * x_i
#     for i in range(n):
#         qii = Qs[i, i]
#         const += 0.5 * qii
#         h[i] += -0.5 * qii

#     # Off-diagonale simmetrica: 2*Qs_ij * x_i x_j per i<j
#     # perché x^TQs x = sum_i Qii x_i + 2 sum_{i<j} Qs_ij x_i x_j
#     for i in range(n):
#         for j in range(i + 1, n):
#             bij = 2.0 * Qs[i, j]  # coeff davanti a x_i x_j
#             const += 0.25 * bij
#             h[i] += -0.25 * bij
#             h[j] += -0.25 * bij
#             Jzz[i, j] += 0.25 * bij

#     # Build SparsePauliOp
#     paulis = []
#     coeffs = []

#     # Costante
#     if abs(const) > tol:
#         paulis.append("I" * n)
#         coeffs.append(const)

#     # Z_i
#     for i in range(n):
#         if abs(h[i]) > tol:
#             label = ["I"] * n
#             label[n - 1 - i] = "Z"   # convenzione Qiskit
#             paulis.append("".join(label))
#             coeffs.append(h[i])

#     # Z_i Z_j
#     for i in range(n):
#         for j in range(i + 1, n):
#             if abs(Jzz[i, j]) > tol:
#                 label = ["I"] * n
#                 label[n - 1 - i] = "Z"
#                 label[n - 1 - j] = "Z"
#                 paulis.append("".join(label))
#                 coeffs.append(Jzz[i, j])

#     if not paulis:
#         paulis = ["I" * n]
#         coeffs = [0.0]

#     return SparsePauliOp(paulis, coeffs=np.array(coeffs, dtype=float))

In [10]:
# import numpy as np

# # E(x)=x^TQx + c
# Q = np.array([
#     [-8.2,  7.2,  6.4,  6.9],
#     [ 0.0, -8.8,  6.7,  6.3],
#     [ 0.0,  0.0, -7.9,  7.0],
#     [ 0.0,  0.0,  0.0, -8.5],
# ], dtype=float)   # esempio upper-triangular
# c = 12.0

# hamiltonian = qubo_to_sparse_pauli(Q, c)
# print(hamiltonian)

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[ 5.425+0.j, -1.025+0.j, -0.65 +0.j, -1.075+0.j, -0.8  +0.j,  1.8  +0.j,
  1.6  +0.j,  1.725+0.j,  1.675+0.j,  1.575+0.j,  1.75 +0.j])
